# Clase 11 — Model Registry con MLflow

Experiment Tracking conserva la evidencia de cada ejecución. Model Registry
organiza los modelos seleccionados para versionarlos, identificarlos y
recuperarlos sin depender de archivos enviados por correo o de un `run_id`
copiado manualmente.

El caso NYC Taxi mantiene las cinco features, el target, el preprocesamiento y
RMSE utilizados en la Clase 10. Cada operación se ejecutará por separado para
observar cómo cambia el estado en MLflow.


> **Quiz 3 — Clases 8, 9 y 10**  
> Contraseña: `mlops2026`

## Antes de comenzar: actualiza el repositorio

Desde la raíz del repositorio público:

```bash
git status
```

Si no hay cambios pendientes y estás en `main`:

```bash
git pull --ff-only origin main
uv sync --locked
```


## Punto de partida

**Prerrequisitos:** distinguir experiment, run, parámetro, métrica, tag y
artifact; reconocer el pipeline de NYC Taxi de la Clase 10; y poder ejecutar
MLflow desde dos terminales.

**Resultado observable:** dos modelos comparables como child runs de un parent
run, dos versiones registradas bajo un nombre estable, los aliases `champion`
y `challenger`, y predicciones obtenidas al cargar cada Model URI.


# 0. Continuación de Experiment Tracking

## 0.1 Flavors e integraciones

Un **flavor** describe cómo MLflow guarda y vuelve a cargar un modelo creado
con una biblioteca determinada.

| Biblioteca | Integración de MLflow |
|---|---|
| scikit-learn | `mlflow.sklearn` |
| XGBoost | `mlflow.xgboost` |
| LightGBM | `mlflow.lightgbm` |
| TensorFlow y Keras | `mlflow.tensorflow` |
| PyTorch | `mlflow.pytorch` |
| Statsmodels | `mlflow.statsmodels` |

Esta práctica conserva `mlflow.sklearn` porque el pipeline vigente usa
scikit-learn. Consulta la [documentación oficial de MLflow Models y sus flavors](https://mlflow.org/docs/latest/ml/model/).

## 0.2 Logging explícito y autologging

`mlflow.sklearn.autolog()` captura información técnica cuando se ejecuta
`.fit(...)`. El RMSE de validación, los periodos de datos y el modelo con su
ejemplo de entrada se registrarán explícitamente porque representan decisiones
del experimento. `log_models=False` evita guardar dos veces el pipeline.
Consulta la [documentación oficial de MLflow Autologging](https://mlflow.org/docs/latest/ml/tracking/autolog/).

## 0.3 Tres recomendaciones de tracking

- registra el contexto que permite interpretar una métrica;
- compara runs con los mismos datos, features y métrica;
- guarda el pipeline completo, no sólo el estimador entrenado.


# Model Registry

![Ciclo de MLOps donde experiment tracking conduce a model versioning y model deployment.](../assets/modulo-02-ciclo-mlops/clase-11/mlops-experiment-tracking-excalidraw.png)

*Experiment Tracking conserva la evidencia de entrenamiento y evaluación;
Model Registry introduce la administración de las versiones seleccionadas
antes de su eventual despliegue.*


# 1. Motivación

<div style="max-width: 280px; margin: 0 auto;">
  <img src="../assets/modulo-02-ciclo-mlops/clase-11/fake-email-anonimizado.png" alt="Correo anonimizado que adjunta directamente un archivo nyc-taxi-model.pkl y solicita desplegarlo." style="width: 100%; height: auto; display: block;">
</div>

Enviar un archivo `pkl` no responde preguntas esenciales:

- ¿qué cambió respecto a la versión anterior?;
- ¿con qué datos, features e hiperparámetros se entrenó?;
- ¿qué preprocesamiento debe ejecutarse?;
- ¿qué ambiente y dependencias necesita?;
- ¿cómo se recupera la versión anterior si aparece una falla?

La captura conserva el ejemplo histórico con datos ficticios.


![Diagrama histórico que conecta modelos guardados en runs del Tracking Server con versiones administradas por Model Registry.](../assets/modulo-02-ciclo-mlops/clase-11/mlflow-tracking-server-model-registry.png)

*Los modelos producidos por runs pueden registrarse como versiones bajo un
nombre estable. En MLflow actual, `champion` y `challenger` son aliases que
apuntan a versiones; `archive` no es una etapa obligatoria.*

## 1.1 Tracking Server y Model Registry

| Tracking Server | Model Registry |
|---|---|
| conserva experiments y runs | organiza modelos seleccionados |
| registra parámetros, métricas, datasets, tags y artifacts | agrupa versiones bajo un nombre estable |
| mantiene también candidatos descartados | contiene los candidatos que se decide registrar |
| responde qué se ejecutó y qué resultado produjo | responde qué versiones existen y qué alias apunta a cada una |

El mismo servidor de MLflow puede exponer ambas responsabilidades y compartir
el Backend Store. Registrar un modelo no lo despliega.


# 2. Definiciones y conceptos

- **MLflow Model:** paquete que contiene el pipeline, su ambiente y la
  información necesaria para cargarlo.
- **Modelo registrado:** nombre estable que agrupa versiones del mismo producto
  predictivo.
- **Versión:** incorporación concreta e inmutable de un MLflow Model al modelo
  registrado.
- **Alias:** nombre mutable que apunta a una versión, como `champion` o
  `challenger`.
- **Tag:** par clave–valor que describe una versión; no redirige una carga.

```text
nyc-taxi-trip-duration
├── versión N     ← alias champion
└── versión N + 1 ← alias challenger
```


## 2.1 URI: identificar recursos con una sintaxis uniforme

**URI** significa **Uniform Resource Identifier**, o identificador uniforme de
recursos. Es una cadena con una sintaxis acordada que identifica un recurso y
comienza con un **esquema**, escrito antes de `:`. El esquema indica al programa
cómo interpretar el resto de la referencia.

| Referencia | Para qué se usa | Ejemplo de la clase |
|---|---|---|
| URL | localiza un recurso de red e indica cómo acceder a él | `http://127.0.0.1:5000` |
| Tracking URI | indica al SDK qué servidor de MLflow recibe las operaciones | `http://127.0.0.1:5000` |
| Run URI | identifica un modelo guardado dentro de un run | `runs:/<run_id>/model` |
| Model URI | identifica una versión o alias dentro de Registry | `models:/nyc-taxi-trip-duration@champion` |
| Ruta local | ubica un archivo para el sistema operativo; no incluye esquema | `labs/trabajo-local/clase-10/mlflow.db` |

Una URL es un tipo de URI orientado a localización en red. `runs:/...` y
`models:/...` no son URLs: MLflow las resuelve mediante sus propias APIs.


# 3. Model Registry — Hands-on

## 3.1 Iniciar el servidor local

En una terminal, entra a la raíz del repositorio e inicia el servidor local:

```bash
uv run mlflow server \
  --backend-store-uri sqlite:///labs/trabajo-local/clase-10/mlflow.db \
  --artifacts-destination ./labs/trabajo-local/clase-10/mlartifacts \
  --host 127.0.0.1 \
  --port 5000
```

Mantén la terminal abierta. La interfaz estará en
`http://127.0.0.1:5000`. Si no carga, confirma que el servidor siga activo y
que otro proceso no esté usando el puerto 5000.


## 3.2 Imports y configuración

Esta celda reúne las bibliotecas y constantes utilizadas durante toda la
práctica. `MODELOS` conserva los dos candidatos conocidos de la Clase 10.


In [ ]:
from pathlib import Path
from pickle import dumps
from time import perf_counter

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
from mlflow import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENTO = "nyc-taxi-clase-11-registry"
MODELO_REGISTRADO = "nyc-taxi-trip-duration"

FEATURES_NUMERICAS = ["distancia_km", "pasajeros", "hora_recoleccion"]
FEATURES_CATEGORICAS = ["zona_origen", "zona_destino"]
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
TARGET = "duracion_minutos"
MODELOS = ("linear_regression", "random_forest")


## 3.3 Conectar el notebook con MLflow

`set_tracking_uri` configura el cliente. `set_experiment` selecciona el
contenedor lógico donde aparecerán los nuevos runs. El resultado muestra el ID
y el nombre del experiment activo.


In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.sklearn.autolog(log_models=False, silent=True)

{
    "experiment_id": experimento.experiment_id,
    "name": experimento.name,
    "tracking_uri": mlflow.get_tracking_uri(),
}


## 3.4 Preparar los datos y el pipeline

Las funciones siguientes usan el mismo contrato de datos y el mismo
preprocesamiento para ambos candidatos. El `ColumnTransformer` queda dentro del
`Pipeline`; por lo tanto, el modelo registrado podrá recibir directamente un
DataFrame con las cinco features.


In [ ]:
def cargar_muestra(ruta):
    viajes = pd.read_csv(ruta)
    enteras = ["pasajeros", "hora_recoleccion", "zona_origen", "zona_destino"]
    viajes[enteras] = viajes[enteras].astype(int)
    return viajes[FEATURES + [TARGET]]


def construir_pipeline(nombre_modelo):
    preprocesamiento = ColumnTransformer(
        [
            ("numericas", "passthrough", FEATURES_NUMERICAS),
            (
                "categoricas",
                OneHotEncoder(handle_unknown="ignore"),
                FEATURES_CATEGORICAS,
            ),
        ]
    )

    if nombre_modelo == "linear_regression":
        estimador = LinearRegression()
    elif nombre_modelo == "random_forest":
        estimador = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
        )
    else:
        raise ValueError(f"Modelo no reconocido: {nombre_modelo}")

    return Pipeline(
        [("preprocesamiento", preprocesamiento), ("modelo", estimador)]
    )


In [ ]:
RAIZ = Path("..").resolve()
DATOS = RAIZ / "labs" / "starters" / "clase-10-experiment-tracking" / "datos"

train = cargar_muestra(DATOS / "green-taxi-train.csv")
valid = cargar_muestra(DATOS / "green-taxi-validation.csv")

pd.DataFrame(
    {
        "periodo": ["2026-03", "2026-04"],
        "uso": ["entrenamiento", "validación"],
        "filas": [len(train), len(valid)],
        "columnas": [len(train.columns), len(valid.columns)],
    }
)


## 3.5 Nested runs: un parent y dos candidatos

El parent run agrupa una pregunta de comparación. Cada child run representa un
candidato completo con sus parámetros, métricas, datasets, tags y MLflow
Model.

```text
comparacion-para-registry
├── linear_regression
└── random_forest
```

La celda conserva `run_id` y `model_uri` en una lista local para usarlos después
sin copiarlos desde la interfaz.


In [ ]:
dataset_train = mlflow.data.from_pandas(
    train,
    source="green-taxi-train.csv",
    targets=TARGET,
    name="green-taxi-2026-03",
)
dataset_valid = mlflow.data.from_pandas(
    valid,
    source="green-taxi-validation.csv",
    targets=TARGET,
    name="green-taxi-2026-04",
)

resultados = []

with mlflow.start_run(run_name="comparacion-para-registry") as parent:
    for nombre_modelo in MODELOS:
        with mlflow.start_run(run_name=nombre_modelo, nested=True) as child:
            pipeline = construir_pipeline(nombre_modelo)
            inicio = perf_counter()
            pipeline.fit(train[FEATURES], train[TARGET])
            tiempo = perf_counter() - inicio
            predicciones = pipeline.predict(valid[FEATURES])
            rmse = root_mean_squared_error(valid[TARGET], predicciones)

            mlflow.log_input(dataset_train, context="training")
            mlflow.log_input(dataset_valid, context="validation")
            mlflow.log_param("model_name", nombre_modelo)
            mlflow.log_metrics(
                {
                    "validation_rmse": rmse,
                    "training_time_seconds": tiempo,
                    "estimated_pickle_size_mib": len(dumps(pipeline)) / (1024**2),
                }
            )
            mlflow.set_tags(
                {
                    "train_month": "2026-03",
                    "validation_month": "2026-04",
                    "candidate_for": MODELO_REGISTRADO,
                }
            )
            modelo = mlflow.sklearn.log_model(
                pipeline,
                name="model",
                input_example=valid[FEATURES].head(5),
                serialization_format=(
                    mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE
                ),
            )
            resultados.append(
                {
                    "model_name": nombre_modelo,
                    "validation_rmse": rmse,
                    "run_id": child.info.run_id,
                    "model_uri": modelo.model_uri,
                }
            )

    mejor = min(resultados, key=lambda resultado: resultado["validation_rmse"])
    mlflow.log_metric("best_validation_rmse", mejor["validation_rmse"])
    mlflow.set_tag("best_child_run_id", mejor["run_id"])
    mlflow.set_tag("parent_run_id", parent.info.run_id)


In [ ]:
resultados_df = (
    pd.DataFrame(resultados)
    .sort_values("validation_rmse")
    .reset_index(drop=True)
)
resultados_df


### ✅ Checkpoint: Tracking antes de Registry

Abre `http://127.0.0.1:5000` y selecciona el experiment
`nyc-taxi-clase-11-registry`. Comprueba:

- un parent run llamado `comparacion-para-registry`;
- dos child runs;
- el RMSE y los parámetros de cada candidato;
- un MLflow Model llamado `model` dentro de cada child run.

No registres todavía desde la interfaz. Primero distingue la evidencia de
Tracking de los objetos que aparecerán en Registry.

> **Seguridad:** los modelos de esta práctica usan serialización `pickle` por
> compatibilidad con `RandomForestRegressor`. Carga únicamente artifacts
> propios o provenientes de una fuente confiable.


# 4. Formas de registrar un modelo

## 4.1 Desde la interfaz

Abre un child run, selecciona su MLflow Model y localiza **Register model**. La
interfaz permite crear un modelo registrado o agregar una versión. No confirmes
la operación durante el recorrido principal para evitar una versión adicional.

## 4.2 Registrar durante `log_model`

Si la decisión ya está tomada al guardar el modelo, puede usarse
`registered_model_name`:

```python
mlflow.sklearn.log_model(
    pipeline,
    name="model",
    registered_model_name=MODELO_REGISTRADO,
    input_example=valid[FEATURES].head(5),
)
```

Esta ruta mezcla el guardado del artifact y el registro. No es la elegida aquí
porque primero queremos comparar ambos runs.

## 4.3 Registrar después de comparar

`mlflow.register_model(model_uri, name)` crea una versión a partir de un MLflow
Model ya guardado. Es la ruta que ejecutaremos porque `resultados_df` ya está
ordenado por RMSE.

## 4.4 API de bajo nivel con `MlflowClient`

También es posible separar explícitamente las operaciones:

```python
client.create_registered_model(name=MODELO_REGISTRADO)
client.create_model_version(
    name=MODELO_REGISTRADO,
    source=model_uri,
    run_id=run_id,
)
```

Esta ruta requiere manejar el caso en que el modelo registrado ya existe. La
usaremos después para administrar aliases, tags y descripciones, no para crear
versiones duplicadas.


## 4.5 Registrar los dos candidatos

Registramos cada candidato por separado con `mlflow.register_model`, la misma
ruta explicada en la sección 4.3, en su propia celda. `resultados_df` ya está
ordenado por RMSE: la primera fila es el mejor candidato y será `champion`; la
segunda es `challenger`.

Cada llamada a `register_model` crea una versión nueva sin comprobar si el run
ya tiene una asociada. Ejecuta estas celdas una sola vez por sesión; repetirlas
genera versiones adicionales innecesarias (ver la sección 6, Errores
frecuentes).

In [ ]:
client = MlflowClient(
    tracking_uri=TRACKING_URI,
    registry_uri=TRACKING_URI,
)

In [ ]:
mejor_candidato = resultados_df.iloc[0]

version_champion = mlflow.register_model(
    model_uri=mejor_candidato["model_uri"],
    name=MODELO_REGISTRADO,
)
version_champion

In [ ]:
segundo_candidato = resultados_df.iloc[1]

version_challenger = mlflow.register_model(
    model_uri=segundo_candidato["model_uri"],
    name=MODELO_REGISTRADO,
)
version_challenger

Con ambas versiones creadas, asignamos `champion` a la de menor RMSE y
`challenger` a la otra. Un alias es una referencia mutable: reasignarlo no crea
una versión nueva, sólo cambia a cuál versión apunta.

In [ ]:
client.set_registered_model_alias(
    name=MODELO_REGISTRADO,
    alias="champion",
    version=version_champion.version,
)
client.set_registered_model_alias(
    name=MODELO_REGISTRADO,
    alias="challenger",
    version=version_challenger.version,
)

Guardamos también un tag con el RMSE de validación y una descripción para cada
versión, tal como se hizo con `update_model_version` el semestre pasado.

In [ ]:
client.set_model_version_tag(
    MODELO_REGISTRADO,
    version_champion.version,
    "validation_rmse",
    f"{mejor_candidato['validation_rmse']:.6f}",
)
client.update_model_version(
    MODELO_REGISTRADO,
    version_champion.version,
    description=(
        f"Pipeline {mejor_candidato['model_name']} validado con abril de 2026."
    ),
)

In [ ]:
client.set_model_version_tag(
    MODELO_REGISTRADO,
    version_challenger.version,
    "validation_rmse",
    f"{segundo_candidato['validation_rmse']:.6f}",
)
client.update_model_version(
    MODELO_REGISTRADO,
    version_challenger.version,
    description=(
        f"Pipeline {segundo_candidato['model_name']} validado con abril de 2026."
    ),
)

### ✅ Checkpoint: versiones y aliases

En **Models**, abre `nyc-taxi-trip-duration` y comprueba que:

- existen dos versiones nuevas;
- `champion` apunta a la de menor RMSE;
- `challenger` apunta a la otra versión;
- cada versión conserva el tag `validation_rmse` y una descripción.


In [ ]:
version_actual_champion = client.get_model_version_by_alias(
    MODELO_REGISTRADO, "champion"
)
version_actual_champion

In [ ]:
version_actual_challenger = client.get_model_version_by_alias(
    MODELO_REGISTRADO, "challenger"
)
version_actual_challenger

In [ ]:
pd.DataFrame(
    [
        {
            "alias": "champion",
            "version": version_actual_champion.version,
            "run_id": version_actual_champion.run_id,
            "description": version_actual_champion.description,
        },
        {
            "alias": "challenger",
            "version": version_actual_challenger.version,
            "run_id": version_actual_challenger.run_id,
            "description": version_actual_challenger.description,
        },
    ]
)

# 5. Cargar y comparar modelos registrados

Una aplicación puede cargar una versión exacta o resolver un alias. El alias
permite mantener estable la URI aunque después apunte a otra versión.


In [ ]:
champion_uri = f"models:/{MODELO_REGISTRADO}@champion"
modelo_champion = mlflow.pyfunc.load_model(champion_uri)
predicciones_champion = modelo_champion.predict(valid[FEATURES])
rmse_champion = root_mean_squared_error(valid[TARGET], predicciones_champion)
rmse_champion

In [ ]:
challenger_uri = f"models:/{MODELO_REGISTRADO}@challenger"
modelo_challenger = mlflow.pyfunc.load_model(challenger_uri)
predicciones_challenger = modelo_challenger.predict(valid[FEATURES])
rmse_challenger = root_mean_squared_error(
    valid[TARGET], predicciones_challenger
)
rmse_challenger

In [ ]:
pd.DataFrame(
    [
        {
            "alias": "champion",
            "model_uri": champion_uri,
            "validation_rmse": rmse_champion,
        },
        {
            "alias": "challenger",
            "model_uri": challenger_uri,
            "validation_rmse": rmse_challenger,
        },
    ]
).sort_values("validation_rmse")

Reasignar un alias cambia qué versión resolverá una Model URI, pero no inicia
ni actualiza un servicio de inferencia. Registry administra referencias y
metadata; deployment es una operación posterior.


# 6. Errores frecuentes

- **La interfaz aparece vacía:** el notebook y el servidor usan Tracking URI
  diferentes.
- **La conexión falla:** el proceso `mlflow server` no está activo en la otra
  terminal.
- **Los child runs aparecen al mismo nivel:** faltó `nested=True` o no había un
  parent run activo.
- **Se creó una versión inesperada:** `mlflow.register_model` no comprueba si
  el run ya tiene una versión asociada; volver a ejecutar la celda de registro
  o confirmar también desde la UI genera una versión adicional.
- **El alias apunta a otra versión:** los aliases son referencias mutables;
  consulta su estado actual con `get_model_version_by_alias`.
- **Se registró sólo el estimador:** el preprocesamiento quedó fuera del
  `Pipeline`.

# 7. Recopilación

- Tracking conserva la evidencia de todos los intentos; Registry organiza los
  modelos seleccionados.
- Una URI identifica un recurso y su esquema indica cómo debe interpretarse.
- Los nested runs agrupan candidatos relacionados sin introducir otra
  herramienta de optimización.
- El pipeline registrado incluye el preprocesamiento y el estimador.
- MLflow permite registrar desde la UI, durante `log_model`, después del run o
  mediante `MlflowClient`; cada ruta responde a un momento de decisión distinto.
- Una versión es inmutable; un alias es una referencia mutable.
- Registrar, promover mediante alias y desplegar son operaciones distintas.


# ✅ Check final de la clase

Comprueba que puedes mostrar y explicar:

- un parent run con dos child runs comparables;
- la tabla local con `run_id`, RMSE y Model URI;
- las cuatro rutas disponibles para registrar un modelo;
- dos versiones bajo `nyc-taxi-trip-duration`;
- los aliases `champion` y `challenger`;
- una predicción obtenida al cargar cada alias.


# Referencias

- [MLflow — Model Registry](https://mlflow.org/docs/latest/ml/model-registry/)
- [MLflow — Model Registry workflow](https://mlflow.org/docs/latest/ml/model-registry/workflow/)
- [MLflow — Autologging](https://mlflow.org/docs/latest/ml/tracking/autolog/)
- [MLflow — Nested runs](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part1-child-runs/)
- [MLflow Python API — `mlflow.register_model`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.register_model)
- [RFC 3986 — Uniform Resource Identifier](https://www.rfc-editor.org/rfc/rfc3986)
